In [1]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("../dataset/upi_transactions_2024.csv")
df.head()
df.info()
df.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 17 columns):
 #   Column              Non-Null Count   Dtype 
---  ------              --------------   ----- 
 0   transaction id      250000 non-null  object
 1   timestamp           250000 non-null  object
 2   transaction type    250000 non-null  object
 3   merchant_category   250000 non-null  object
 4   amount (INR)        250000 non-null  int64 
 5   transaction_status  250000 non-null  object
 6   sender_age_group    250000 non-null  object
 7   receiver_age_group  250000 non-null  object
 8   sender_state        250000 non-null  object
 9   sender_bank         250000 non-null  object
 10  receiver_bank       250000 non-null  object
 11  device_type         250000 non-null  object
 12  network_type        250000 non-null  object
 13  fraud_flag          250000 non-null  int64 
 14  hour_of_day         250000 non-null  int64 
 15  day_of_week         250000 non-null  object
 16  is

(250000, 17)

In [4]:
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
      .str.replace("(", "", regex=False)
      .str.replace(")", "", regex=False)
)
df.columns

Index(['transaction_id', 'timestamp', 'transaction_type', 'merchant_category',
       'amount_inr', 'transaction_status', 'sender_age_group',
       'receiver_age_group', 'sender_state', 'sender_bank', 'receiver_bank',
       'device_type', 'network_type', 'fraud_flag', 'hour_of_day',
       'day_of_week', 'is_weekend'],
      dtype='object')

In [5]:
df.isnull().sum()
missing = (df.isnull().sum() / len(df)) * 100
missing.sort_values(ascending=False)

transaction_id        0.0
sender_bank           0.0
day_of_week           0.0
hour_of_day           0.0
fraud_flag            0.0
network_type          0.0
device_type           0.0
receiver_bank         0.0
sender_state          0.0
timestamp             0.0
receiver_age_group    0.0
sender_age_group      0.0
transaction_status    0.0
amount_inr            0.0
merchant_category     0.0
transaction_type      0.0
is_weekend            0.0
dtype: float64

Numeric Columns

In [6]:
numeric_cols = df.select_dtypes(include=np.number).columns

for col in numeric_cols:
    df[col].fillna(df[col].median(), inplace=True)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_12496\2579964100.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)


Categorical Columns

In [7]:
cat_cols = df.select_dtypes(include="object").columns

for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_12496\719105301.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mode()[0], inplace=True)


Check Duplicates

In [8]:
df.duplicated().sum()
df.drop_duplicates(inplace=True)
df.duplicated().sum()

np.int64(0)

In [9]:
df["timestamp"] = pd.to_datetime(df["timestamp"])
df["timestamp"].head()
df["year"] = df["timestamp"].dt.year
df["month"] = df["timestamp"].dt.month
df["month_name"] = df["timestamp"].dt.month_name()
df["day"] = df["timestamp"].dt.day

In [10]:
(df["amount_inr"] < 0).sum()
df = df[df["amount_inr"] > 0]
df["amount_inr"].describe()
df["transaction_status"].value_counts()
df["fraud_flag"].value_counts()

fraud_flag
0    249520
1       480
Name: count, dtype: int64

Outlier

In [11]:
Q1 = df["amount_inr"].quantile(0.25)
Q3 = df["amount_inr"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[
    (df["amount_inr"] < lower) |
    (df["amount_inr"] > upper)
]

print("Outliers:", len(outliers))

Outliers: 21171


In [12]:
df.info()
df.describe(include="all")
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 21 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   transaction_id      250000 non-null  object        
 1   timestamp           250000 non-null  datetime64[ns]
 2   transaction_type    250000 non-null  object        
 3   merchant_category   250000 non-null  object        
 4   amount_inr          250000 non-null  int64         
 5   transaction_status  250000 non-null  object        
 6   sender_age_group    250000 non-null  object        
 7   receiver_age_group  250000 non-null  object        
 8   sender_state        250000 non-null  object        
 9   sender_bank         250000 non-null  object        
 10  receiver_bank       250000 non-null  object        
 11  device_type         250000 non-null  object        
 12  network_type        250000 non-null  object        
 13  fraud_flag          250000 no

,transaction_id,timestamp,transaction_type,merchant_category,amount_inr,transaction_status,sender_age_group,receiver_age_group,sender_state,sender_bank,...,device_type,network_type,fraud_flag,hour_of_day,day_of_week,is_weekend,year,month,month_name,day
0,TXN0000000001,2024-10-08 15:17:28,P2P,Entertainment,868,SUCCESS,26-35,18-25,Delhi,Axis,...,Android,4G,0,15,Tuesday,0,2024,10,October,8
1,TXN0000000002,2024-04-11 06:56:00,P2M,Grocery,1011,SUCCESS,26-35,26-35,Uttar Pradesh,ICICI,...,iOS,4G,0,6,Thursday,0,2024,4,April,11
2,TXN0000000003,2024-04-02 13:27:18,P2P,Grocery,477,SUCCESS,26-35,36-45,Karnataka,Yes Bank,...,Android,4G,0,13,Tuesday,0,2024,4,April,2
3,TXN0000000004,2024-01-07 10:09:17,P2P,Fuel,2784,SUCCESS,26-35,26-35,Delhi,ICICI,...,Android,5G,0,10,Sunday,1,2024,1,January,7
4,TXN0000000005,2024-01-23 19:04:23,P2P,Shopping,990,SUCCESS,26-35,18-25,Delhi,Axis,...,iOS,WiFi,0,19,Tuesday,0,2024,1,January,23


In [13]:
df.to_csv(
    "../dataset/cleaned_transactions.csv",
    index=False
)